# OpenML · Phase 1 — training telemetry
Trains a small torch MLP and fans metrics out to **MLflow** (:5000), **Aim** (:43800),
and **Pushgateway → Grafana** (:3000 → *ML Training* dashboard) live, then registers
the model in the MLflow Model Registry.

Requires the **track** and **monitor** stacks to be ON (toggle them in the console at :8080).

In [ ]:
import time, numpy as np, torch, torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import openml_telemetry as tel

X, y = make_classification(n_samples=4000, n_features=20, n_informative=12, n_classes=3, random_state=0)
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, random_state=0)

# publish data-prep stats -> Grafana 'Data Prep' dashboard
import collections
tel.push_dataset_stats('synth3', rows=len(X),
    classes=dict(collections.Counter(y.tolist())),
    splits={'train': len(Xtr), 'val': len(Xval)})
print('dataset pushed:', len(X), 'rows')

In [ ]:
Xtr_t = torch.tensor(Xtr, dtype=torch.float32); ytr_t = torch.tensor(ytr)
Xval_t = torch.tensor(Xval, dtype=torch.float32); yval_t = torch.tensor(yval)

model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.92)
loss_fn = nn.CrossEntropyLoss()
EPOCHS = 20

with tel.TrainingRun('synth3-mlp', params={'lr': 1e-2, 'epochs': EPOCHS, 'opt': 'adam'}) as run:
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad()
        out = model(Xtr_t); loss = loss_fn(out, ytr_t)
        loss.backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            acc = (model(Xval_t).argmax(1) == yval_t).float().mean().item()
        run.log(step=epoch, epoch=epoch, loss=loss.item(), accuracy=acc,
                lr=opt.param_groups[0]['lr'])
        print(f'epoch {epoch:2d}  loss={loss.item():.4f}  val_acc={acc:.3f}')
        time.sleep(0.5)   # so the live Grafana gauges visibly move
    info = run.register_torch(model, 'synth3-mlp')
print('done')

### Where to look
* **MLflow** http://localhost:5000 — run params/metrics + the registered model `synth3-mlp`
* **Aim** http://localhost:43800 — compare runs
* **Grafana** http://localhost:3000 → *ML Training* — the live loss/accuracy/lr you just streamed
* **Grafana** → *Host & Containers* — CPU/RAM of every OpenML container